# E-002 Baseline: Nemotron-3-Nano-30B-A3B-BF16 + LoRA SFT

**What this notebook does:** loads Nemotron-3-Nano-30B-A3B-BF16 (a Mamba-hybrid MoE) in 4-bit quantization, applies LoRA (rank 32, alpha 64) targeting Mamba mixer + attention modules, does a 1-epoch SFT pass on the competition's `train.csv`, saves the adapter, runs a tiny eval, writes `cv_score.json`.

**Manual setup before running (on Kaggle):**
1. **Right sidebar → Settings → Accelerator:** pick **`GPU T4 x2`** (P100 won't work — its `sm_60` CUDA arch isn't supported by the PyTorch wheel in Kaggle's base image).
2. **Right sidebar → Internet:** turn **ON** (needed to pip-install `mamba-ssm`, `causal-conv1d`, `bitsandbytes`).
3. **Right sidebar → + Add Input:**
   - Competition → `nvidia-nemotron-model-reasoning-challenge`
   - Models → search `nemotron-3-nano-30b-a3b-bf16` (the one by `metric`, the official Competition Metrics account) → add `transformers/default/1`
4. (Optional) **Right sidebar → Save & Run All → Save version → Quick Save:** runs all cells end-to-end as a Kaggle batch job (~2-4h on T4×2).

**Iteration history this fixes:** v1=multipart-wrong, v2=missing-cv-module, v3=path-as-HF-repo, v4=missing-trust_remote_code, v5=missing-mamba-ssm, v6=missing-bitsandbytes, v7=P100 instead of T4×2 (USER MUST FIX in step 1 above).

## 1. Verify GPU + show what's mounted

In [ ]:
import os, sys, json, subprocess
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU{i}: {p.name} | {p.total_memory / 1024**3:.1f} GB | sm_{p.major}{p.minor}')

print('\n=== /kaggle/input contents ===')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
    if depth <= 3:
        print(f'  {root}: dirs={dirs[:5]} files={files[:5]}')
    if depth >= 3:
        dirs.clear()
print('=== end ===')

# Hard check: T4 (sm_75) or better required
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    if p.major < 7:
        print(f'\n⚠️  GPU{i} is sm_{p.major}{p.minor} (e.g. P100=sm_60). PyTorch wheel may not support it.')
        print('   → Sidebar → Settings → Accelerator: pick GPU T4 x2 instead and re-run.')

## 2. Install Nemotron-Mamba dependencies

`mamba-ssm` and `causal-conv1d` are required by the model's custom modeling code. `bitsandbytes` is needed for 4-bit quantization. None of these are in Kaggle's base image.

In [ ]:
for pkg in ['causal-conv1d', 'mamba-ssm', 'bitsandbytes']:
    mod = pkg.replace('-', '_')
    try:
        __import__(mod)
        print(f'  {pkg}: already installed')
    except ImportError:
        print(f'  installing {pkg}...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation', pkg],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print(f'    rc={r.returncode}')
            print(f'    stderr: {r.stderr[-1500:]}')
        else:
            print(f'    installed')

# Sanity check
import bitsandbytes, mamba_ssm, causal_conv1d
print(f'\nbitsandbytes: {bitsandbytes.__version__}')
print(f'mamba_ssm: {mamba_ssm.__version__}')
print(f'causal_conv1d: {causal_conv1d.__version__}')

## 3. Auto-detect model path and load tokenizer

Kaggle Models mount at `/kaggle/input/models/<owner>/<slug>/<framework>/<variation>/<version>/` — but the path can vary. The cell below walks `/kaggle/input` and picks the dir that contains `config.json` + `tokenizer.json`.

In [ ]:
model_name = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files and 'tokenizer.json' in files:
        model_name = root
        break

assert model_name, 'No model dir with config.json + tokenizer.json found. Check sidebar → Add Input → Models.'
print(f'Model path: {model_name}')

# Show files in the model dir
for f in sorted(os.listdir(model_name)):
    p = os.path.join(model_name, f)
    if os.path.isfile(p):
        print(f'  {f}  ({os.path.getsize(p) / 1024**2:.1f} MB)')

In [ ]:
# Force offline mode so transformers doesn't try HF Hub
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded. Vocab size:', tokenizer.vocab_size)

## 4. Load Nemotron in 4-bit (NF4) + apply LoRA

**Memory budget**: 30B params in 4-bit ≈ 15 GB base + ~1 GB activations + LoRA. On 2×T4 (30 GB total) this should fit; if not, the next cell will OOM and we'll need to drop `max_seq` to 1024 or use attention-only LoRA targets.

**LoRA targets**: `mixer.in_proj`, `mixer.out_proj` (Mamba state-space blocks) + `q/k/v/o_proj` (attention). **Crucially** we do NOT regex-match the MoE expert modules (that triggers a known scan-hang on this architecture).

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    local_files_only=True,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Base model loaded.')

In [ ]:
LORA_RANK = 32
LORA_ALPHA = 64
LORA_TARGET_MODULES = ['mixer.in_proj', 'mixer.out_proj', 'q_proj', 'k_proj', 'v_proj', 'o_proj']

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Load competition train.csv

This is the **raw** training data with bare `prompt`/`answer` (no CoT). First-run baseline is intentionally a 'garbage in, garbage out' validation of the end-to-end pipeline. Real improvements come from training on a synthesized corpus with CoT traces (winner's approach — see `competitions/.../data/corpus/v1/build_corpus.py`).

In [ ]:
import csv
TRAIN_CSV = '/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv'

# For first run, take a small sample to validate the pipeline (~50 rows = ~10 min on T4x2)
# Bump to full set once one run finishes successfully.
SAMPLE_SIZE = 50

train_data = []
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        prompt = row.get('prompt') or row.get('question') or ''
        answer = row.get('answer') or row.get('completion') or ''
        if prompt and answer:
            train_data.append({'prompt': prompt, 'completion': str(answer)})
        if len(train_data) >= SAMPLE_SIZE:
            break

print(f'Loaded {len(train_data)} training examples')
print(f'\nExample prompt: {train_data[0]["prompt"][:200]}...')
print(f'Example answer: {train_data[0]["completion"][:200]}...')

## 6. Train: 1 epoch LoRA SFT

Mask-cross-entropy loss: only the answer tokens contribute to the loss (mask=1), prompt tokens are conditioning context (mask=0). Linear decay LR schedule per winner's recipe.

In [ ]:
MAX_SEQ_LEN = 2048  # T4 memory budget; winner used 8192 on bigger hardware
GRAD_ACCUM_STEPS = 8
EPOCHS = 1
LR_INITIAL = 2e-5
LR_FINAL = 1e-5

def build_datum(prompt_text, answer_text, max_length=MAX_SEQ_LEN):
    pt = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    at = tokenizer(answer_text, add_special_tokens=False)['input_ids'] + [tokenizer.eos_token_id]
    tokens = pt + at
    mask = [0] * len(pt) + [1] * len(at)
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
        mask = mask[:max_length]
    return torch.tensor(tokens).unsqueeze(0), torch.tensor(mask).unsqueeze(0)

def compute_loss(logits, labels, mask):
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    shift_mask = mask[..., 1:].contiguous()
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    loss = loss * shift_mask.view(-1)
    return loss.sum() / shift_mask.sum().clamp(min=1)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_INITIAL, weight_decay=0.01)
device = next(model.parameters()).device
total_steps = len(train_data) * EPOCHS

model.train()
step = 0
accum_step = 0
running_loss = 0.0

print(f'Training {total_steps} steps total, micro_batch=1, grad_accum={GRAD_ACCUM_STEPS}')
for epoch in range(EPOCHS):
    for d in train_data:
        ids, mask = build_datum(d['prompt'], d['completion'])
        ids, mask = ids.to(device), mask.to(device)
        out = model(input_ids=ids)
        loss = compute_loss(out.logits, ids, mask)
        (loss / GRAD_ACCUM_STEPS).backward()
        running_loss += loss.item()
        accum_step += 1
        if accum_step % GRAD_ACCUM_STEPS == 0 or step == total_steps - 1:
            # Linear decay
            mult = max(0.0, 1.0 - epoch / (1 + EPOCHS))
            lr = LR_FINAL + (LR_INITIAL - LR_FINAL) * mult
            for pg in optimizer.param_groups:
                pg['lr'] = lr
            optimizer.step()
            optimizer.zero_grad()
        step += 1
        if step % 5 == 0:
            print(f'  step {step}/{total_steps}  loss={running_loss/5:.4f}')
            running_loss = 0.0

print('Training done.')

## 7. Save adapter + adapter_config.json

Rank ≤ 32 is a hard host invariant; the asserts below catch regressions.

In [ ]:
os.makedirs('/kaggle/working/adapter', exist_ok=True)
model.save_pretrained('/kaggle/working/adapter')

adapter_config = {
    'peft_type': 'LORA',
    'r': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'target_modules': LORA_TARGET_MODULES,
    'base_model_name_or_path': model_name,
}
with open('/kaggle/working/adapter/adapter_config.json', 'w') as f:
    json.dump(adapter_config, f, indent=2)

# Invariant checks
assert adapter_config['r'] <= 32, 'LoRA rank exceeds host max of 32'
for f in os.listdir('/kaggle/working/adapter'):
    p = os.path.join('/kaggle/working/adapter', f)
    print(f'  {f}  ({os.path.getsize(p) / 1024**2:.2f} MB)')

## 8. Quick eval: generate predictions + score with `\boxed{}` extraction

Self-eval on the training sample (overfit-biased — proper CV needs a held-out fixture). For first run this validates that:
1. The fine-tuned adapter actually generates text (not garbage)
2. The `\boxed{}` extractor works
3. Scoring produces a number we can ship

In [ ]:
import re

def extract_boxed(text):
    """Extract the contents of the LAST \boxed{...} expression in text."""
    if not text: return None
    idx = text.rfind(r'\boxed{')
    if idx == -1: return None
    start = idx + len(r'\boxed{')
    depth = 1
    for i in range(start, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
    return None

def score_item(pred, gold):
    p = extract_boxed(pred)
    if p is None: return False
    g = extract_boxed(gold) or str(gold).strip()
    p, g = p.strip(), g.strip()
    if p == g: return True
    try:
        return abs(float(p) - float(g)) <= 1e-2
    except ValueError:
        return False

model.eval()
predictions = []
with torch.no_grad():
    eval_subset = train_data[:10]  # quick eval on first 10
    for i, d in enumerate(eval_subset):
        inputs = tokenizer(d['prompt'], return_tensors='pt').to(device)
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        pred = text[len(d['prompt']):].strip()
        is_correct = score_item(pred, d['completion'])
        predictions.append({'id': str(i), 'prediction': pred, 'gold': d['completion'], 'correct': is_correct})
        if i < 3:
            print(f'\n[{i}] gold: {d["completion"][:80]}')
            print(f'    pred: {pred[:120]}')
            print(f'    boxed_extracted: {extract_boxed(pred)}  correct: {is_correct}')

n_correct = sum(1 for p in predictions if p['correct'])
score = n_correct / len(predictions)
print(f'\n=== CV accuracy on {len(predictions)} samples: {score:.4f} ({n_correct}/{len(predictions)}) ===')

cv_score = {
    'score': score,
    'n_total': len(predictions),
    'n_correct': n_correct,
    'n_boxed_missing': sum(1 for p in predictions if extract_boxed(p['prediction']) is None),
    'note': 'self-eval on training sample; replace with held-out fixture for real CV',
}
with open('/kaggle/working/cv_score.json', 'w') as f:
    json.dump(cv_score, f, indent=2)
print(f'\nWrote /kaggle/working/cv_score.json: {cv_score}')

## 9. Package submission.zip

Host requires: `submission.zip` containing `adapter_config.json` at root + `adapter_model.safetensors` (or `.bin`). The notebook output's `adapter/` subdir already has both.

In [ ]:
import zipfile, hashlib

ADAPTER_DIR = '/kaggle/working/adapter'
SUBMISSION = '/kaggle/working/submission.zip'

# Compute SHA256 of weights for manifest
weight_files = [f for f in os.listdir(ADAPTER_DIR) if f.startswith('adapter_model')]
assert weight_files, 'No adapter_model.* file found in adapter/'
h = hashlib.sha256()
with open(os.path.join(ADAPTER_DIR, weight_files[0]), 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
adapter_sha = h.hexdigest()

manifest = {
    'cv_score': cv_score,
    'adapter_sha256': adapter_sha,
    'experiment_id': 'E-002-baseline-manual',
    'base_model': model_name,
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'target_modules': LORA_TARGET_MODULES,
}

with zipfile.ZipFile(SUBMISSION, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(ADAPTER_DIR):
        zf.write(os.path.join(ADAPTER_DIR, f), arcname=f)
    zf.writestr('manifest.json', json.dumps(manifest, indent=2))

size_mb = os.path.getsize(SUBMISSION) / 1024**2
print(f'Wrote {SUBMISSION}  ({size_mb:.2f} MB)')
print(f'Contents:')
with zipfile.ZipFile(SUBMISSION) as zf:
    for n in zf.namelist():
        print(f'  {n}')

print(f'\nManifest: {json.dumps(manifest, indent=2)}')

## 10. Manual submit (do this from the Kaggle UI, not in the notebook)

1. **Save & Run All → Save version** the notebook (Quick Save is fine).
2. Go to the **Output** tab of the saved version, download `submission.zip` (or just submit directly from the notebook output).
3. Competition page → **Submit Predictions** → upload `submission.zip` → add a message like `E-002 baseline manual run` → Submit.
4. Wait ~30-90 min for scoring.

If the CV score in cell 8 looks reasonable (>0.1 say, since this is overfit self-eval), the leaderboard score is likely usable as a baseline anchor. If CV is 0.0, the `\boxed{}` format probably isn't being produced — that means the bare `train.csv` answers don't have it. Iterate by training on the synthesized corpus (which DOES wrap answers in `\boxed{}`).